# Hierarchical Concept Bottleneck Model on ViT-B/16 — CUB-200-2011

**Architecture:** identical hierarchy to `02_H-CBM ResNet-50_train.ipynb`, but the
backbone is **ViT-B/16 (SWAG E2E V1, 384×384)** instead of ResNet-50.

```
Image (384×384) ──► ViT-B/16 SWAG ──► f ∈ ℝ⁷⁶⁸
                                       │
                              ┌────────┴────────┐
                              ▼                 ▼
                    coarse_head g_c      (concat with p_c)
                    768 → 256 → 13       fine_head g_f
                       z_c, p_c          (768+13) → 512 → 312
                                              │  soft hierarchical mask:
                                              │  p_f[i] = σ(z_f[i]) · (0.5 + 0.5·p_c[parent(i)])
                                              ▼
                                          classifier h
                                          312 → 200      ◀ strict bottleneck
```

**Training (3 phases — same recipe as ResNet R4):**

| Phase | Trainable                                        | Loss                                       | Notes                                  |
|-------|--------------------------------------------------|--------------------------------------------|----------------------------------------|
| 1     | coarse + fine heads + classifier (backbone frozen) | `λ_c L_c + λ_f L_f + 0.1 L_task`        | joint head training                    |
| 2     | classifier only (heads & backbone frozen)        | `L_task` on **predicted** `p_f`            | distribution-matched calibration       |
| 3     | **everything** (backbone unfrozen)               | `2 L_c + 2 L_f + L_task` + MixUp(α=0.2)   | warmup + cosine, overfit guard, save by best `val_acc` |

**GPU optimisations (tuned for NVIDIA H100 80GB):** cudnn.benchmark, TF32
matmul, BF16 AMP (Hopper-native, no GradScaler), Flash-Attention v2 SDPA,
`torch.compile(mode='max-autotune')`, BATCH_SIZE=96, 12 dataloader workers,
prefetch_factor=4, pinned memory, persistent workers.

**ViT-specific tuning:** `BACKBONE_LR = 1e-5` for P1/P2 and `3e-6` for P3 —
SWAG E2E was pretrained on Instagram, so the backbone genuinely needs to
adapt to fine-grained CUB birds.


## 1. Imports, Data Pipeline, Transforms

In [1]:
# ─── Cell 1: Imports, Drive, Data Pipeline ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, random, pickle, time
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm
from torchvision.models import ViT_B_16_Weights
from PIL import Image
import matplotlib.pyplot as plt

# ── Paths (same convention as file 02) ───────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/XAI-Project/DB/DB1'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
PKL_DIR    = f'{DRIVE_ROOT}/pipeline'
DATA_PKL   = f'{PKL_DIR}/data_pipeline_vit.pkl'   # IMG_SIZE=384
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PKL_DIR,  exist_ok=True)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU   : {p.name}  ({p.total_memory/1e9:.0f} GB)')
    print(f'BF16  : {torch.cuda.is_bf16_supported()}   '
          f'TF32 : {torch.backends.cuda.matmul.allow_tf32}')

# ── Load pipeline ────────────────────────────────────────────────────────────
with open(DATA_PKL, 'rb') as f:
    bundle = pickle.load(f)
paths, splits, meta = bundle['paths'], bundle['splits'], bundle['meta']
DATASET_DIR     = paths['dataset_dir']
NUM_L1          = meta['NUM_L1']
NUM_L2          = meta['NUM_L2']
CONCEPT_NAMES   = meta['CONCEPT_NAMES']
attr_parent_idx = np.asarray(meta['attr_parent_idx'])
IMG_SIZE        = int(meta.get('IMG_SIZE', 384))

# ── H100-tuned batch & loader settings ──────────────────────────────────────
# H100 80GB HBM3: ViT-B/16 SWAG (86M backbone) trainable @ 384×384 with BF16 AMP
# fits batch 96 comfortably (~55-60 GB peak). Batch 128 works only when backbone
# is frozen. Stick with 96 for safety across Phase 3 (full unfreeze).
BATCH_SIZE   = 96
NUM_WORKERS  = 12       # Colab H100 nodes expose ~12 vCPU
PREFETCH     = 4

# ── Transforms (same as file 02 + 384 input) ─────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
])
eval_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 256 / 224)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class BirdDataset(Dataset):
    def __init__(self, sp, root, tf):
        self.imgs = sp['images']; self.lbl = sp['labels']
        self.l1   = np.asarray(sp['l1'],   dtype=np.float32)
        self.l2   = np.asarray(sp['l2'],   dtype=np.float32)
        self.cert = np.asarray(sp['cert'], dtype=np.float32)
        self.vis  = np.asarray(sp['vis'],  dtype=np.float32)
        self.root = root; self.tf = tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, 'images', self.imgs[i])).convert('RGB')
        return (self.tf(img),
                torch.tensor(self.lbl[i] - 1, dtype=torch.long),
                torch.from_numpy(self.l1[i]),
                torch.from_numpy(self.l2[i]),
                torch.from_numpy(self.cert[i]),
                torch.from_numpy(self.vis[i]))

train_ds = BirdDataset(splits['train'], DATASET_DIR, train_transform)
val_ds   = BirdDataset(splits['val'],   DATASET_DIR, eval_transform)
test_ds  = BirdDataset(splits['test'],  DATASET_DIR, eval_transform)

dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
             persistent_workers=True, prefetch_factor=PREFETCH, drop_last=False)
train_loader = DataLoader(train_ds, shuffle=True,  **dl_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **dl_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kw)

# Per-part stats for L_coarse pos_weight (Cell 4)
train_l1  = train_ds.l1
train_vis = train_ds.vis

print(f'IMG_SIZE={IMG_SIZE}  BATCH_SIZE={BATCH_SIZE}  NUM_L1={NUM_L1}  NUM_L2={NUM_L2}')
print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


Mounted at /content/drive
Device: cuda
GPU   : NVIDIA RTX PRO 6000 Blackwell Server Edition  (102 GB)
BF16  : True   TF32 : False
IMG_SIZE=384  BATCH_SIZE=96  NUM_L1=13  NUM_L2=312
Train: 5,394 | Val: 600 | Test: 5,794
Train batches: 57 | Val batches: 7


## 2. Define the H-CBM Architecture (ViT-B/16 backbone)

Identical class structure to file 02's `HierarchicalCBM`, with three differences:

1. **Backbone**: `vit_b_16(IMAGENET1K_SWAG_E2E_V1)` instead of `resnet50(IMAGENET1K_V2)`.
2. **Feature dim**: 768 (ViT CLS token) instead of 2048.
3. **Input size**: 384×384 instead of 224×224.

The forward pass, hierarchical masking, and `detach_coarse` flag are byte-for-byte
the same logic — only the feature extractor changes.


In [2]:
# ─── Cell 2: HierarchicalCBM on ViT-B/16 ────────────────────────────────────
class HierarchicalCBM(nn.Module):
    """Hierarchical Concept Bottleneck Model — same logic as file 02, ViT backbone."""

    def __init__(self, attr_parent_idx, num_classes=200, num_l1=13, num_l2=312):
        super().__init__()
        self.num_l1 = num_l1
        self.num_l2 = num_l2
        self.register_buffer(
            'attr_parent_idx',
            torch.tensor(attr_parent_idx, dtype=torch.long))

        # ViT-B/16 SWAG E2E V1 — strongest publicly available ViT-B
        # (~85.3% ImageNet top-1 vs ~81% for vanilla ViT-B/16)
        weights  = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
        backbone = tvm.vit_b_16(weights=weights)
        backbone.heads = nn.Identity()              # output: 768-d CLS token
        self.features = backbone

        # Coarse head: 768 → 256 → 13
        # Smaller hidden than file 02 (which had 2048→256) — ViT features are
        # denser per dim, so 256 hidden is plenty for 13 binary outputs.
        self.coarse_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_l1),
        )

        # Fine head: (768 + num_l1) → 512 → 312  (same as file 02, just 2048→768)
        self.fine_head = nn.Sequential(
            nn.Linear(768 + num_l1, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_l2),
        )

        # Strict bottleneck classifier
        self.classifier = nn.Linear(num_l2, num_classes)

    def forward(self, x, detach_coarse: bool = False):
        feats = self.features(x)                          # (B, 768)
        z_c   = self.coarse_head(feats)
        p_c   = torch.sigmoid(z_c)

        # Phase-1 trick: stop L_fine gradient from flowing back through p_c
        p_c_for_fine = p_c.detach() if detach_coarse else p_c
        z_f = self.fine_head(torch.cat([feats, p_c_for_fine], dim=1))
        p_f_raw = torch.sigmoid(z_f)

        # Soft hierarchical mask: mask ∈ [0.5, 1.0] instead of [0, 1].
        # Preserves the hierarchy signal but prevents p_f → 0 early in
        # training when p_c ≈ 0.5, which otherwise starves the classifier.
        parent_idx = self.attr_parent_idx
        has_parent = parent_idx >= 0
        safe_idx   = parent_idx.clamp(min=0)
        parent_p   = p_c[:, safe_idx]
        soft_parent = 0.5 + 0.5 * parent_p
        mask = torch.where(has_parent.unsqueeze(0),
                           soft_parent, torch.ones_like(parent_p))
        p_f = p_f_raw * mask

        cls_logits = self.classifier(p_f)
        return cls_logits, z_c, p_c, z_f, p_f


model = HierarchicalCBM(
    attr_parent_idx=attr_parent_idx,
    num_classes=200, num_l1=NUM_L1, num_l2=NUM_L2,
).to(DEVICE)

n_back = sum(p.numel() for p in model.features.parameters())
n_head = sum(p.numel() for n, p in model.named_parameters()
             if not n.startswith('features'))
print(f'HierarchicalCBM (ViT-B/16 SWAG):')
print(f'  Backbone params : {n_back:>11,d}  ({n_back/1e6:.1f}M)')
print(f'  Head params     : {n_head:>11,d}  ({n_head/1e6:.2f}M)')
print(f'  Total           : {n_back+n_head:>11,d}')

# Sanity check (and verify detach_coarse does not change forward output)
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out_n = model(dummy, detach_coarse=False)
    out_d = model(dummy, detach_coarse=True)
    assert torch.allclose(out_n[0], out_d[0]), 'detach_coarse changed output!'
print(f'Forward OK: cls={tuple(out_n[0].shape)}, '
      f'z_c={tuple(out_n[1].shape)}, z_f={tuple(out_n[3].shape)}')

print('detach_coarse sanity check passed.')
model.train()

Downloading: "https://download.pytorch.org/models/vit_b_16_swag-9ac1b537.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16_swag-9ac1b537.pth


100%|██████████| 331M/331M [00:00<00:00, 476MB/s] 


HierarchicalCBM (ViT-B/16 SWAG):
  Backbone params :  86,090,496  (86.1M)
  Head params     :     824,781  (0.82M)
  Total           :  86,915,277
Forward OK: cls=(2, 200), z_c=(2, 13), z_f=(2, 312)
detach_coarse sanity check passed.


HierarchicalCBM(
  (features): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          (ln_

## 3. Training Configuration (ViT-tuned)

In [3]:
# ─── Cell 3: Hyperparameters (ViT-B/16, target 80%+) ────────────────────────
CKPT_PATH    = os.path.join(CKPT_DIR, 'best_hcbm_vit_384.pth')
HISTORY_PATH = os.path.join(PKL_DIR,  'hcbm_vit_history.pkl')
CKPT_1_PATH  = os.path.join(CKPT_DIR, 'hcbm_vit_p1_best.pth')
CKPT_2_PATH  = os.path.join(CKPT_DIR, 'hcbm_vit_p2_best.pth')

# ── Learning rates ─────────────────────────────────────────────────────────
# BACKBONE_LR=1e-5 in P1/P2: the previous run used 1e-6, which kept SWAG
#   features effectively frozen — that is why val_acc stalled at 38%.
#   CUB-200 (fine-grained birds) is far enough from Instagram (SWAG
#   pretraining) that the backbone genuinely needs to adapt.
BACKBONE_LR        = 1e-5
HEADS_LR           = 3e-4

# Phase-3 LRs are 3× lower (everything is unfrozen → easy to overfit).
PHASE3_BACKBONE_LR = 3e-6
PHASE3_HEADS_LR    = 1e-4

WEIGHT_DECAY       = 5e-4
PHASE3_WD          = 1e-3

# ── Loss weights ───────────────────────────────────────────────────────────
LAMBDA_COARSE      = 1.0
LAMBDA_FINE        = 1.0
LAMBDA_TASK        = 1.0
# Small task signal during Phase-1 joint head training so the concept space
# is shaped to be discriminative for classification from the start.
LAMBDA_TASK_P1     = 0.1

# Phase-3 concept-dominant weights (×2): keeps the concept space
# well-calibrated while the classifier is refined end-to-end.
PHASE3_LAMBDA_COARSE = 2.0
PHASE3_LAMBDA_FINE   = 2.0
PHASE3_LAMBDA_TASK   = 1.0

# ── Stability ──────────────────────────────────────────────────────────────
LABEL_SMOOTHING    = 0.1
GRAD_CLIP          = 1.0
WARMUP_EPOCHS      = 5
MIXUP_ALPHA        = 0.2     # MixUp regulariser for Phase 3 only

# ── Plateau / patience ─────────────────────────────────────────────────────
LR_PATIENCE        = 7
LR_FACTOR          = 0.3
PHASE1_PATIENCE    = 20
EARLY_STOP_PAT     = 40
OVERFIT_THRESHOLD  = 0.20
OVERFIT_PATIENCE   = 15

# ── Max epochs (EarlyStopper governs the actual stop) ──────────────────────
PHASE1_EPOCHS   = 80
PHASE2_EPOCHS   = 40
PHASE3_EPOCHS   = 150

print(f'CKPT_PATH        : {CKPT_PATH}')
print(f'HISTORY_PATH     : {HISTORY_PATH}')
print(f'Backbone lr: P1/P2={BACKBONE_LR}  P3={PHASE3_BACKBONE_LR}')
print(f'Heads    lr: P1/P2={HEADS_LR}  P3={PHASE3_HEADS_LR}')
print(f'WD: P1/P2={WEIGHT_DECAY}  P3={PHASE3_WD}')
print(f'λ_coarse: P1/P3={LAMBDA_COARSE}/{PHASE3_LAMBDA_COARSE}  '
      f'λ_fine: P1/P3={LAMBDA_FINE}/{PHASE3_LAMBDA_FINE}  '
      f'λ_task: P1/P3={LAMBDA_TASK_P1}/{PHASE3_LAMBDA_TASK}')
print(f'Label smoothing={LABEL_SMOOTHING}  Grad clip={GRAD_CLIP}  '
      f'Warmup={WARMUP_EPOCHS} ep  MixUp α={MIXUP_ALPHA}')
print(f'Max epochs: P1={PHASE1_EPOCHS}  P2={PHASE2_EPOCHS}  P3={PHASE3_EPOCHS}')
print(f'ES patience: P1={PHASE1_PATIENCE}  P2&P3={EARLY_STOP_PAT}')
print(f'Overfit threshold={OVERFIT_THRESHOLD} ({OVERFIT_PATIENCE} ep)')


CKPT_PATH        : /content/drive/MyDrive/XAI-Project/DB/DB1/checkpoints/best_hcbm_vit_384.pth
HISTORY_PATH     : /content/drive/MyDrive/XAI-Project/DB/DB1/pipeline/hcbm_vit_history.pkl
Backbone lr: P1/P2=1e-05  P3=3e-06
Heads    lr: P1/P2=0.0003  P3=0.0001
WD: P1/P2=0.0005  P3=0.001
λ_coarse: P1/P3=1.0/2.0  λ_fine: P1/P3=1.0/2.0  λ_task: P1/P3=0.1/1.0
Label smoothing=0.1  Grad clip=1.0  Warmup=5 ep  MixUp α=0.2
Max epochs: P1=80  P2=40  P3=150
ES patience: P1=20  P2&P3=40
Overfit threshold=0.2 (15 ep)


## 4. Loss Functions

`L_total = λ_c · L_coarse + λ_f · L_fine + λ_t · L_task`

| Loss | Formula | Masking |
|------|---------|---------|
| `L_coarse` | Weighted BCE on `z_c` vs L1 targets        | part visibility |
| `L_fine`   | Focal Loss (α=0.25, γ=2) on `z_f` vs L2     | certainty ≥ 3   |
| `L_task`   | Cross-entropy on `cls_logits`               | none            |


In [ ]:
# ─── Cell 4: FocalLoss + compute_loss + per-part pos_weights ────────────────
class FocalLoss(nn.Module):
    """Focal loss for class imbalance in binary attribute prediction."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets, mask=None):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt  = torch.exp(-bce)
        f   = self.alpha * (1 - pt) ** self.gamma * bce
        if mask is not None:
            return (f * mask).sum() / mask.sum().clamp(min=1)
        return f.mean()


def compute_loss(cls_out, z_c, p_c, z_f, p_f,
                 labels, l1, l2, cert, vis,
                 focal_fn, l1_pw, lam_c, lam_f, lam_t,
                 label_smoothing=0.0):
    l_c = F.binary_cross_entropy_with_logits(z_c, l1, pos_weight=l1_pw, reduction='none')
    l_c = (l_c * vis).sum() / vis.sum().clamp(min=1)
    l_f = focal_fn(z_f, l2, mask=cert)
    l_t = F.cross_entropy(cls_out, labels, label_smoothing=label_smoothing)
    return lam_c*l_c + lam_f*l_f + lam_t*l_t, l_c, l_f, l_t


# ── Per-part positive weights from train split ──
l1_pos_weights_list = []
for j in range(NUM_L1):
    vj = train_vis[:, j];  aj = train_l1[:, j]
    n_pos = float(((aj == 1) & (vj == 1)).sum())
    n_neg = float(((aj == 0) & (vj == 1)).sum())
    l1_pos_weights_list.append(n_neg / max(n_pos, 1.0))

l1_pos_weights = torch.tensor(l1_pos_weights_list, dtype=torch.float32, device=DEVICE)
focal_loss_fn  = FocalLoss(alpha=0.25, gamma=2.0)

print('L_coarse pos_weights (per-part, N_neg/N_pos):')
for name, w in zip(CONCEPT_NAMES[:NUM_L1], l1_pos_weights.tolist()):
    print(f'  {name:14s}: {w:.2f}')


L_coarse pos_weights (per-part, N_neg/N_pos):
  back          : 0.06
  belly         : 0.04
  bill          : 0.01
  breast        : 0.06
  crown         : 0.05
  eye           : 0.05
  forehead      : 0.05
  head          : 0.05
  leg           : 0.08
  nape          : 0.07
  tail          : 0.09
  throat        : 0.04
  wing          : 0.05


## 5. Training Utilities

In [5]:
# ─── Cell 5: Train / Eval helpers + EarlyStopper ────────────────────────────

# ── H100 AMP: BF16 native (Hopper). No GradScaler needed for BF16. ─────────
_AMP_DTYPE = torch.bfloat16
torch.set_float32_matmul_precision('high')   # TF32 for fp32 matmuls on H100
# Enable SDPA flash-attention backend (Hopper has FP16/BF16 flash attn v2)
try:
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)
except Exception:
    pass
print(f'AMP dtype: {_AMP_DTYPE}  | matmul precision: high  | flash SDPA: on')


def _orig(m):
    """Return un-compiled module (after torch.compile wraps it)."""
    return m._orig_mod if hasattr(m, '_orig_mod') else m


def set_trainable(model, backbone, coarse, fine, classifier):
    m = _orig(model)
    for p in m.features.parameters():    p.requires_grad = backbone
    for p in m.coarse_head.parameters(): p.requires_grad = coarse
    for p in m.fine_head.parameters():   p.requires_grad = fine
    for p in m.classifier.parameters():  p.requires_grad = classifier


def make_optimizer(model, backbone_lr, heads_lr, weight_decay):
    m = _orig(model)
    bb = [p for p in m.features.parameters() if p.requires_grad]
    hd = [p for n, p in m.named_parameters()
          if p.requires_grad and not n.startswith('features')]
    groups = []
    if bb: groups.append({'params': bb, 'lr': backbone_lr})
    if hd: groups.append({'params': hd, 'lr': heads_lr})
    return torch.optim.AdamW(groups, weight_decay=weight_decay)


def make_plateau(opt, patience, factor):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=patience, factor=factor)


class EarlyStopper:
    """Same semantics as file 02: val_loss patience + optional overfit guard."""
    def __init__(self, patience=20, min_delta=1e-4,
                 overfit_threshold=None, overfit_patience=10):
        self.patience = patience; self.min_delta = min_delta
        self.oft = overfit_threshold; self.ofp = overfit_patience
        self.best = float('inf')
        self.cnt = 0; self.ofcnt = 0
        self.improved = False; self.stop = False; self.reason = ''

    def step(self, val_loss, train_acc=None, val_acc=None):
        if val_loss < self.best - self.min_delta:
            self.best = val_loss; self.cnt = 0; self.improved = True
        else:
            self.cnt += 1; self.improved = False
            if self.cnt >= self.patience:
                self.stop = True
                self.reason = f'no val_loss improvement for {self.patience} epochs'
        if (self.oft is not None and train_acc is not None and val_acc is not None):
            gap = train_acc - val_acc
            if gap > self.oft:
                self.ofcnt += 1
                if self.ofcnt >= self.ofp:
                    self.stop = True
                    self.reason = (f'overfitting: gap={gap:.3f} > {self.oft} '
                                   f'for {self.ofp} epochs')
            else:
                self.ofcnt = 0


def train_one_epoch(model, loader, opt, focal_fn, l1_pw, device,
                    lam_c, lam_f, lam_t, label_smoothing=0.0,
                    grad_clip=None, detach_coarse=False, scaler=None):
    model.train()
    sum_loss = sum_c = sum_f = sum_t = 0.0
    correct = total = 0
    use_amp = (scaler is not None and device.type == 'cuda')
    for imgs, lbl, l1, l2, cert, vis in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device,  non_blocking=True)
        l1   = l1.to(device,   non_blocking=True)
        l2   = l2.to(device,   non_blocking=True)
        cert = cert.to(device, non_blocking=True)
        vis  = vis.to(device,  non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=use_amp):
            cls_out, zc, pc, zf, pf = model(imgs, detach_coarse=detach_coarse)
            loss, lc, lf, lt = compute_loss(
                cls_out, zc, pc, zf, pf, lbl, l1, l2, cert, vis,
                focal_fn, l1_pw, lam_c, lam_f, lam_t,
                label_smoothing=label_smoothing)
        if use_amp and scaler.is_enabled():
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip)
            opt.step()
        bs = lbl.size(0)
        sum_loss += loss.item() * bs
        sum_c += lc.item() * bs;  sum_f += lf.item() * bs;  sum_t += lt.item() * bs
        correct += (cls_out.argmax(1) == lbl).sum().item();  total += bs
    return {'loss':   sum_loss / total, 'acc':    correct / total,
            'coarse': sum_c    / total, 'fine':   sum_f    / total,
            'task':   sum_t    / total}


@torch.no_grad()
def evaluate(model, loader, focal_fn, l1_pw, device,
             lam_c, lam_f, lam_t, label_smoothing=0.0):
    model.eval()
    sum_loss = sum_c = sum_f = sum_t = 0.0
    correct = total = 0
    use_amp = device.type == 'cuda'
    for imgs, lbl, l1, l2, cert, vis in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device,  non_blocking=True)
        l1   = l1.to(device,   non_blocking=True)
        l2   = l2.to(device,   non_blocking=True)
        cert = cert.to(device, non_blocking=True)
        vis  = vis.to(device,  non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=use_amp):
            cls_out, zc, pc, zf, pf = model(imgs)
            loss, lc, lf, lt = compute_loss(
                cls_out, zc, pc, zf, pf, lbl, l1, l2, cert, vis,
                focal_fn, l1_pw, lam_c, lam_f, lam_t,
                label_smoothing=label_smoothing)
        bs = lbl.size(0)
        sum_loss += loss.item() * bs
        sum_c += lc.item() * bs;  sum_f += lf.item() * bs;  sum_t += lt.item() * bs
        correct += (cls_out.argmax(1) == lbl).sum().item();  total += bs
    return {'loss':   sum_loss / total, 'acc':    correct / total,
            'coarse': sum_c    / total, 'fine':   sum_f    / total,
            'task':   sum_t    / total}


def train_phase2_epoch(model, loader, opt, device, label_smoothing=0.0, scaler=None):
    """
    Phase 2 (calibration): train classifier on the model's PREDICTED p_f
    (concept heads + backbone frozen).

    Rationale: in Phase 3 the classifier sees predicted p_f (continuous,
    attenuated by the hierarchical mask), not GT l2 ∈ {0,1}. Training on
    GT l2 here would create a distribution mismatch and collapse Phase 3
    accuracy — that is the bug the previous ViT run hit (val_acc=38%).
    """
    model.eval()                            # heads + backbone frozen
    _orig(model).classifier.train()         # only classifier trainable
    sum_loss = correct = total = 0
    use_amp = (scaler is not None and device.type == 'cuda')
    for imgs, lbl, l1, l2, cert, vis in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device,  non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=use_amp):
            with torch.no_grad():
                _, _, _, _, p_f = model(imgs)        # predicted concepts
            out  = _orig(model).classifier(p_f)      # classifier on predicted p_f
            loss = F.cross_entropy(out, lbl, label_smoothing=label_smoothing)
        if use_amp and scaler.is_enabled():
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            loss.backward(); opt.step()
        bs = lbl.size(0)
        sum_loss += loss.item() * bs
        correct += (out.argmax(1) == lbl).sum().item();  total += bs
    return {'loss': sum_loss/total, 'acc': correct/total}


@torch.no_grad()
def eval_phase2(model, loader, device):
    """Phase 2 eval: classifier on PREDICTED p_f (matches Phase-3 input)."""
    model.eval();  sum_loss = correct = total = 0
    use_amp = device.type == 'cuda'
    for imgs, lbl, l1, l2, cert, vis in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device,  non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=use_amp):
            _, _, _, _, p_f = model(imgs)
            out  = _orig(model).classifier(p_f)
            loss = F.cross_entropy(out, lbl)
        bs = lbl.size(0)
        sum_loss += loss.item() * bs
        correct += (out.argmax(1) == lbl).sum().item();  total += bs
    return {'loss': sum_loss/total, 'acc': correct/total}


# ── MixUp helper for Phase 3 ───────────────────────────────────────────────
def _mixup_batch(imgs, lbl, l1, l2, cert, vis, alpha):
    """Linearly blend pairs of images and concept targets; return both labels."""
    if alpha <= 0:
        return imgs, lbl, l1, l2, cert, vis, lbl, 1.0
    lam  = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    imgs = lam * imgs + (1 - lam) * imgs[perm]
    l1   = lam * l1   + (1 - lam) * l1[perm]
    l2   = lam * l2   + (1 - lam) * l2[perm]
    # Binary masks: take element-wise max so a concept is supervised whenever
    # EITHER source image had it visible / certain.
    cert = torch.maximum(cert, cert[perm])
    vis  = torch.maximum(vis,  vis[perm])
    return imgs, lbl, l1, l2, cert, vis, lbl[perm], lam


def train_p3_mixup_epoch(model, loader, opt, focal_fn, l1_pw, device,
                          lam_c, lam_f, lam_t, label_smoothing=0.0,
                          grad_clip=None, scaler=None, mixup_alpha=0.2):
    """Phase-3 epoch with MixUp on (image, l1, l2) and label-mixed CE."""
    model.train()
    sum_loss = sum_c = sum_f = sum_t = 0.0
    correct = total = 0
    use_amp = (scaler is not None and device.type == 'cuda')
    for imgs, lbl, l1, l2, cert, vis in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device,  non_blocking=True)
        l1   = l1.to(device,   non_blocking=True)
        l2   = l2.to(device,   non_blocking=True)
        cert = cert.to(device, non_blocking=True)
        vis  = vis.to(device,  non_blocking=True)
        imgs_m, l_a, l1_m, l2_m, cert_m, vis_m, l_b, lam = _mixup_batch(
            imgs, lbl, l1, l2, cert, vis, mixup_alpha)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=use_amp):
            cls_out, zc, pc, zf, pf = model(imgs_m, detach_coarse=False)
            # Concept losses on blended targets (continuous in [0,1]):
            loss_blend, lc, lf, _ = compute_loss(
                cls_out, zc, pc, zf, pf, l_a, l1_m, l2_m, cert_m, vis_m,
                focal_fn, l1_pw, lam_c, lam_f, 0.0, label_smoothing=0.0)
            # MixUp CE on the task: linear blend of two cross-entropies
            ce_a = F.cross_entropy(cls_out, l_a, label_smoothing=label_smoothing)
            ce_b = F.cross_entropy(cls_out, l_b, label_smoothing=label_smoothing)
            l_t  = lam * ce_a + (1 - lam) * ce_b
            loss = loss_blend + lam_t * l_t
        if use_amp and scaler.is_enabled():
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip)
            opt.step()
        bs = lbl.size(0)
        sum_loss += loss.item() * bs
        sum_c += lc.item() * bs;  sum_f += lf.item() * bs;  sum_t += l_t.item() * bs
        preds = cls_out.argmax(1)
        correct += (lam * (preds == l_a).float()
                    + (1 - lam) * (preds == l_b).float()).sum().item()
        total   += bs
    return {'loss':   sum_loss / total, 'acc':    correct / total,
            'coarse': sum_c    / total, 'fine':   sum_f    / total,
            'task':   sum_t    / total}


# ── Verify on a dummy batch ─────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    db_imgs = torch.randn(4, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    db_lbl  = torch.randint(0, 200, (4,), device=DEVICE)
    db_l1   = torch.randint(0, 2, (4, NUM_L1), dtype=torch.float32, device=DEVICE)
    db_l2   = torch.randint(0, 2, (4, NUM_L2), dtype=torch.float32, device=DEVICE)
    db_cert = torch.randint(0, 2, (4, NUM_L2), dtype=torch.float32, device=DEVICE)
    db_vis  = torch.ones(4, NUM_L1, device=DEVICE)
    cls_o, zc, pc, zf, pf = model(db_imgs)
    L, lc, lf, lt = compute_loss(cls_o, zc, pc, zf, pf,
                                  db_lbl, db_l1, db_l2, db_cert, db_vis,
                                  focal_loss_fn, l1_pos_weights,
                                  LAMBDA_COARSE, LAMBDA_FINE, LAMBDA_TASK,
                                  label_smoothing=LABEL_SMOOTHING)
print(f'Dummy losses: total={L.item():.4f}  c={lc:.4f}  f={lf:.4f}  t={lt:.4f}')
print('Utilities defined.')
model.train()


AMP dtype: torch.bfloat16  | matmul precision: high  | flash SDPA: on
Dummy losses: total=5.8820  c=0.3871  f=0.0464  t=5.4485
Utilities defined.


HierarchicalCBM(
  (features): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          (ln_

## 6. Run Training (3 phases — same recipe as ResNet R4)

| Phase | Trainable                                   | Loss                                | Max Ep         | Stopping                       |
|-------|---------------------------------------------|-------------------------------------|----------------|--------------------------------|
| 1     | coarse + fine heads + classifier (bb frozen)| `λ_c L_c + λ_f L_f + 0.1 L_task`    | ≤PHASE1_EPOCHS | val_loss plateau               |
| 2     | classifier only (heads frozen, **predicted p_f** input) | `L_task`                | ≤PHASE2_EPOCHS | val_loss plateau               |
| 3     | **all** (backbone unfrozen)                 | `2 L_c + 2 L_f + L_task` + MixUp(α=0.2) | ≤PHASE3_EPOCHS | val_loss + overfit guard       |

**Best checkpoint is saved by `val_acc`** (concept-dominant losses make val_loss
non-monotonic with accuracy in Phase 3).


GPU optimisations applied: `cudnn.benchmark`, TF32, BF16 AMP, `torch.compile`.

In [6]:
# ─── Cell 6: 3-Phase Training (mirrors ResNet R4 recipe) ────────────────────

# ── GPU maximisation ────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark         = True
torch.backends.cudnn.allow_tf32        = True
torch.backends.cuda.matmul.allow_tf32  = True
torch.backends.cudnn.deterministic     = False

# H100 uses BF16 → no loss scaling needed. Pass a disabled scaler so the
# train_one_epoch() AMP branch still runs (it falls through to plain backward).
_scaler = torch.cuda.amp.GradScaler(enabled=False)

def _ts(): return datetime.now().strftime('[%H:%M:%S]')
def _banner(title):
    print(); print('=' * 72); print(f'{_ts()}  {title}'); print('=' * 72)

print(f'{_ts()}  cudnn.benchmark=True  TF32=True  GradScaler={_scaler.is_enabled()}')

# torch.compile — Hopper benefits a lot from max-autotune.
try:
    if hasattr(torch, 'compile') and DEVICE.type == 'cuda':
        model = torch.compile(model, mode='max-autotune', fullgraph=False)
        print(f'{_ts()}  torch.compile(mode="max-autotune") applied')
except Exception as e:
    print(f'{_ts()}  torch.compile skipped: {e}')

# ── History (resume-friendly; old 4-phase schema is migrated to 3-phase) ───
_default_history = {
    'phase1':  {'train_loss': [], 'val_loss': [],
                'train_coarse': [], 'val_coarse': [],
                'train_fine':   [], 'val_fine':   []},
    'phase2':  {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []},
    'phase3':  {'train_loss': [], 'val_loss': [],
                'train_acc':  [], 'val_acc':  [],
                'train_coarse': [], 'val_coarse': [],
                'train_fine':   [], 'val_fine':   [],
                'train_task':   [], 'val_task':   []},
}
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, 'rb') as fh:
        history = pickle.load(fh)
    if 'phase1' not in history:
        history['phase1'] = _default_history['phase1']
        history.pop('phase1a', None); history.pop('phase1b', None)
        print(f'{_ts()}  Migrated history schema: phase1a/phase1b → phase1 (cleared)')
    for ph, d in _default_history.items():
        history.setdefault(ph, d)
    print(f'{_ts()}  Resumed history from {HISTORY_PATH}')
else:
    history = _default_history

def _save_hist():
    with open(HISTORY_PATH, 'wb') as fh: pickle.dump(history, fh)

def _save_ckpt(path, ep, vlm, extra=None):
    sd = _orig(model).state_dict()
    payload = {'epoch': ep, 'model_state_dict': sd}
    if extra: payload.update(extra)
    payload.update(vlm)
    torch.save(payload, path)


# ══════════════════════════════════════════════════════════════════════════
#  PHASE 1 — Joint head training (backbone frozen; coarse + fine + cls on)
# ══════════════════════════════════════════════════════════════════════════
_banner(f'Phase 1 — Joint heads  (≤{PHASE1_EPOCHS} ep | ES pat={PHASE1_PATIENCE})')
set_trainable(model, backbone=False, coarse=True, fine=True, classifier=True)
opt_1   = make_optimizer(model, BACKBONE_LR, HEADS_LR, WEIGHT_DECAY)
sch_1   = make_plateau(opt_1, LR_PATIENCE, LR_FACTOR)
es_1    = EarlyStopper(patience=PHASE1_PATIENCE)

for ep in range(1, PHASE1_EPOCHS + 1):
    t0  = time.time()
    trm = train_one_epoch(model, train_loader, opt_1,
                          focal_loss_fn, l1_pos_weights, DEVICE,
                          lam_c=LAMBDA_COARSE, lam_f=LAMBDA_FINE, lam_t=LAMBDA_TASK_P1,
                          label_smoothing=LABEL_SMOOTHING,
                          grad_clip=GRAD_CLIP, detach_coarse=False, scaler=_scaler)
    vlm = evaluate(model, val_loader, focal_loss_fn, l1_pos_weights, DEVICE,
                   lam_c=LAMBDA_COARSE, lam_f=LAMBDA_FINE, lam_t=LAMBDA_TASK_P1,
                   label_smoothing=LABEL_SMOOTHING)
    tr_loss = trm['coarse'] + trm['fine'] + LAMBDA_TASK_P1 * trm['task']
    vl_loss = vlm['coarse'] + vlm['fine'] + LAMBDA_TASK_P1 * vlm['task']
    sch_1.step(vl_loss)
    es_1.step(vl_loss)
    history['phase1']['train_loss'].append(tr_loss)
    history['phase1']['val_loss'].append(vl_loss)
    history['phase1']['train_coarse'].append(trm['coarse'])
    history['phase1']['val_coarse'].append(vlm['coarse'])
    history['phase1']['train_fine'].append(trm['fine'])
    history['phase1']['val_fine'].append(vlm['fine'])
    mark = ' ✓' if es_1.improved else f' ({es_1.cnt}/{PHASE1_PATIENCE})'
    print(f'{_ts()}  P1 {ep:3d}/{PHASE1_EPOCHS} | '
          f'coarse tr={trm["coarse"]:.4f} vl={vlm["coarse"]:.4f} | '
          f'fine tr={trm["fine"]:.4f} vl={vlm["fine"]:.4f} | '
          f'lr={opt_1.param_groups[-1]["lr"]:.1e} | '
          f'{time.time()-t0:.0f}s{mark}')
    if es_1.improved:
        _save_ckpt(CKPT_1_PATH, ep,
                   {'val_coarse': vlm['coarse'], 'val_fine': vlm['fine']})
    if es_1.stop:
        print(f'{_ts()}  [Early stop P1] {es_1.reason}'); break

c1 = torch.load(CKPT_1_PATH, map_location=DEVICE, weights_only=False)
_orig(model).load_state_dict(c1['model_state_dict'])
print(f'{_ts()}  → loaded best P1  ep={c1["epoch"]}  '
      f'val_coarse={c1["val_coarse"]:.4f}  val_fine={c1["val_fine"]:.4f}')
_save_hist()


# ══════════════════════════════════════════════════════════════════════════
#  PHASE 2 — Classifier calibration (heads frozen; predicted p_f input)
# ══════════════════════════════════════════════════════════════════════════
_banner(f'Phase 2 — Classifier calibration  (≤{PHASE2_EPOCHS} ep | ES pat={EARLY_STOP_PAT})')
set_trainable(model, backbone=False, coarse=False, fine=False, classifier=True)
opt_2   = make_optimizer(model, BACKBONE_LR, HEADS_LR, WEIGHT_DECAY)
sch_2   = make_plateau(opt_2, LR_PATIENCE, LR_FACTOR)
es_2    = EarlyStopper(patience=EARLY_STOP_PAT)

for ep in range(1, PHASE2_EPOCHS + 1):
    t0  = time.time()
    trm = train_phase2_epoch(model, train_loader, opt_2, DEVICE,
                              label_smoothing=LABEL_SMOOTHING, scaler=_scaler)
    vlm = eval_phase2(model, val_loader, DEVICE)
    sch_2.step(vlm['loss'])
    es_2.step(vlm['loss'])
    history['phase2']['train_loss'].append(trm['loss'])
    history['phase2']['val_loss'].append(vlm['loss'])
    history['phase2']['train_acc'].append(trm['acc'])
    history['phase2']['val_acc'].append(vlm['acc'])
    mark = ' ✓' if es_2.improved else f' ({es_2.cnt}/{EARLY_STOP_PAT})'
    print(f'{_ts()}  P2 {ep:3d}/{PHASE2_EPOCHS} | '
          f'tr loss={trm["loss"]:.4f} acc={trm["acc"]:.3f} | '
          f'vl loss={vlm["loss"]:.4f} acc={vlm["acc"]:.3f} | '
          f'lr={opt_2.param_groups[-1]["lr"]:.1e} | '
          f'{time.time()-t0:.0f}s{mark}')
    if es_2.improved:
        _save_ckpt(CKPT_2_PATH, ep, {'val_loss': vlm['loss'], 'val_acc': vlm['acc']})
    if es_2.stop:
        print(f'{_ts()}  [Early stop P2] {es_2.reason}'); break

c2 = torch.load(CKPT_2_PATH, map_location=DEVICE, weights_only=False)
_orig(model).load_state_dict(c2['model_state_dict'])
print(f'{_ts()}  → loaded best P2  ep={c2["epoch"]}  '
      f'val_loss={c2["val_loss"]:.4f}  val_acc={c2["val_acc"]:.3f}')
_save_hist()


# ══════════════════════════════════════════════════════════════════════════
#  PHASE 3 — Joint fine-tune (everything unfrozen) + MixUp
# ══════════════════════════════════════════════════════════════════════════
_banner(f'Phase 3 — Joint fine-tune  (≤{PHASE3_EPOCHS} ep | ES pat={EARLY_STOP_PAT} '
        f'| OFT={OVERFIT_THRESHOLD})')
set_trainable(model, backbone=True, coarse=True, fine=True, classifier=True)
optimizer = make_optimizer(model, PHASE3_BACKBONE_LR, PHASE3_HEADS_LR, PHASE3_WD)

_warm = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
_cos  = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(PHASE3_EPOCHS - WARMUP_EPOCHS, 1), eta_min=1e-7)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[_warm, _cos], milestones=[WARMUP_EPOCHS])
es_3 = EarlyStopper(
    patience=EARLY_STOP_PAT,
    overfit_threshold=OVERFIT_THRESHOLD, overfit_patience=OVERFIT_PATIENCE)

# Best-val_acc tracking (concept-dominant losses make val_loss non-monotonic
# with val_acc; we save by best val_acc but keep ES on val_loss).
best_val_acc_p3 = -1.0
best_acc_epoch  = 0

for ep in range(1, PHASE3_EPOCHS + 1):
    t0  = time.time()
    trm = train_p3_mixup_epoch(model, train_loader, optimizer,
                                focal_loss_fn, l1_pos_weights, DEVICE,
                                PHASE3_LAMBDA_COARSE, PHASE3_LAMBDA_FINE, PHASE3_LAMBDA_TASK,
                                label_smoothing=LABEL_SMOOTHING,
                                grad_clip=GRAD_CLIP, scaler=_scaler,
                                mixup_alpha=MIXUP_ALPHA)
    vlm = evaluate(model, val_loader, focal_loss_fn, l1_pos_weights, DEVICE,
                   PHASE3_LAMBDA_COARSE, PHASE3_LAMBDA_FINE, PHASE3_LAMBDA_TASK,
                   label_smoothing=LABEL_SMOOTHING)
    scheduler.step()
    es_3.step(vlm['loss'], train_acc=trm['acc'], val_acc=vlm['acc'])

    for k, v in [
        ('train_loss',   trm['loss']),    ('val_loss',   vlm['loss']),
        ('train_acc',    trm['acc']),     ('val_acc',    vlm['acc']),
        ('train_coarse', trm['coarse']),  ('val_coarse', vlm['coarse']),
        ('train_fine',   trm['fine']),    ('val_fine',   vlm['fine']),
        ('train_task',   trm['task']),    ('val_task',   vlm['task']),
    ]:
        history['phase3'][k].append(v)

    acc_improved = vlm['acc'] > best_val_acc_p3
    if acc_improved:
        best_val_acc_p3 = vlm['acc']
        best_acc_epoch  = ep

    gap   = trm['acc'] - vlm['acc']
    lmark = ' ✓L' if es_3.improved else f' ({es_3.cnt}/{EARLY_STOP_PAT})'
    amark = ' ✓A' if acc_improved else ''
    print(f'{_ts()}  Best val_acc={best_val_acc_p3:.4f} @ ep{best_acc_epoch}')
    print(f'{_ts()}  P3 {ep:3d}/{PHASE3_EPOCHS} | '
          f'tr loss={trm["loss"]:.4f} acc={trm["acc"]:.3f} '
          f'(c={trm["coarse"]:.3f} f={trm["fine"]:.3f} t={trm["task"]:.3f}) | '
          f'vl loss={vlm["loss"]:.4f} acc={vlm["acc"]:.3f} | '
          f'gap={gap:.3f}({es_3.ofcnt}/{OVERFIT_PATIENCE}) | '
          f'lr={optimizer.param_groups[-1]["lr"]:.2e} | '
          f'{time.time()-t0:.0f}s{lmark}{amark}')

    # Checkpoint on best val_acc (the metric that actually matters here)
    if acc_improved:
        _save_ckpt(CKPT_PATH, ep,
                   {'val_loss': vlm['loss'], 'val_acc': vlm['acc']},
                   extra={'attr_parent_idx': attr_parent_idx,
                          'CONCEPT_NAMES':   CONCEPT_NAMES,
                          'NUM_L1':          NUM_L1,
                          'NUM_L2':          NUM_L2,
                          'IMG_SIZE':        IMG_SIZE,
                          'arch':            'HierarchicalCBM-ViT-B16-SWAG'})
        _save_hist()

    if es_3.stop:
        print(f'{_ts()}  [Early stop P3] {es_3.reason}'); break

_save_hist()
print()
print(f'{_ts()}  ✓ Training complete.')
print(f'{_ts()}  Best Phase-3 val_acc  : {best_val_acc_p3:.4f} @ ep{best_acc_epoch}')
print(f'{_ts()}  Best Phase-3 val_loss : {es_3.best:.4f}')
print(f'{_ts()}  Checkpoint            : {CKPT_PATH}')
print(f'{_ts()}  History               : {HISTORY_PATH}')


[03:00:13]  cudnn.benchmark=True  TF32=True  GradScaler=False


/tmp/ipykernel_838/1175830893.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  _scaler = torch.cuda.amp.GradScaler(enabled=False)


[03:00:15]  torch.compile(mode="max-autotune") applied
[03:00:15]  Migrated history schema: phase1a/phase1b → phase1 (cleared)
[03:00:15]  Resumed history from /content/drive/MyDrive/XAI-Project/DB/DB1/pipeline/hcbm_vit_history.pkl

[03:00:15]  Phase 1 — Joint heads  (≤80 ep | ES pat=20)


/usr/local/lib/python3.12/dist-packages/torch/_inductor/select_algorithm.py:3686: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  current_out_size = out_base.storage().size()
E0603 03:01:20.526000 838 torch/_inductor/select_algorithm.py:3924] [0/0] Runtime error during autotuning: 
E0603 03:01:20.526000 838 torch/_inductor/select_algorithm.py:3924] [0/0] No valid triton configs. OutOfMemoryError: out of resource: triton_mm Required: 131072 Hardware limit:101376 Reducing block sizes or `num_stages` may help.. 
E0603 03:01:20.526000 838 torch/_inductor/select_algorithm.py:3924] [0/0] Ignoring this choice.
E0603 03:01:20.596000 838 torch/_inductor/select_algorithm.py:3924] [0/0] Runtime error during autotuning: 
E0603 03:01:20.596000 838 torch/_inductor/

[03:06:42]  P1   1/80 | coarse tr=0.0710 vl=0.0711 | fine tr=0.0267 vl=0.0184 | lr=3.0e-04 | 387s ✓
[03:06:50]  P1   2/80 | coarse tr=0.0672 vl=0.0704 | fine tr=0.0207 vl=0.0195 | lr=3.0e-04 | 7s ✓
[03:06:58]  P1   3/80 | coarse tr=0.0654 vl=0.0702 | fine tr=0.0231 vl=0.0232 | lr=3.0e-04 | 8s ✓
[03:07:06]  P1   4/80 | coarse tr=0.0641 vl=0.0709 | fine tr=0.0275 vl=0.0282 | lr=3.0e-04 | 7s ✓
[03:07:14]  P1   5/80 | coarse tr=0.0632 vl=0.0722 | fine tr=0.0329 vl=0.0335 | lr=3.0e-04 | 8s ✓
[03:07:22]  P1   6/80 | coarse tr=0.0624 vl=0.0736 | fine tr=0.0381 vl=0.0394 | lr=3.0e-04 | 8s ✓
[03:07:30]  P1   7/80 | coarse tr=0.0632 vl=0.0775 | fine tr=0.0428 vl=0.0435 | lr=3.0e-04 | 8s ✓
[03:07:38]  P1   8/80 | coarse tr=0.0621 vl=0.0780 | fine tr=0.0468 vl=0.0468 | lr=3.0e-04 | 7s ✓
[03:07:46]  P1   9/80 | coarse tr=0.0605 vl=0.0829 | fine tr=0.0496 vl=0.0501 | lr=3.0e-04 | 8s ✓
[03:07:54]  P1  10/80 | coarse tr=0.0600 vl=0.0807 | fine tr=0.0516 vl=0.0512 | lr=3.0e-04 | 8s ✓
[03:08:02]  P1  11

## 7. Best Checkpoint Summary

In [7]:
# ─── Cell 7: Best checkpoint summary ────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)

print('Best checkpoint summary:')
print(f'  arch     : {ckpt.get("arch", "HierarchicalCBM-ViT-B16-SWAG")}')
print(f'  epoch    : {ckpt["epoch"]}')
print(f'  val_loss : {ckpt["val_loss"]:.4f}')
print(f'  val_acc  : {ckpt["val_acc"]:.4f}  ({ckpt["val_acc"]*100:.1f}%)')
print(f'  IMG_SIZE : {ckpt["IMG_SIZE"]}   '
      f'NUM_L1={ckpt["NUM_L1"]}  NUM_L2={ckpt["NUM_L2"]}')

print('\nTo load this model in another notebook:')
print('  ckpt  = torch.load(CKPT_PATH, weights_only=False)')
print('  model = HierarchicalCBM(ckpt["attr_parent_idx"], 200,')
print('                          ckpt["NUM_L1"], ckpt["NUM_L2"]).to(DEVICE)')
print('  model.load_state_dict(ckpt["model_state_dict"])')


Best checkpoint summary:
  arch     : HierarchicalCBM-ViT-B16-SWAG
  epoch    : 66
  val_loss : 1.7045
  val_acc  : 0.8650  (86.5%)
  IMG_SIZE : 384   NUM_L1=13  NUM_L2=312

To load this model in another notebook:
  ckpt  = torch.load(CKPT_PATH, weights_only=False)
  model = HierarchicalCBM(ckpt["attr_parent_idx"], 200,
                          ckpt["NUM_L1"], ckpt["NUM_L2"]).to(DEVICE)
  model.load_state_dict(ckpt["model_state_dict"])
